In [0]:
from datetime import datetime

daily_summary = spark.sql("""
    SELECT
        SUM(total_transactions)     AS total_txns,
        ROUND(SUM(total_amount), 2) AS total_amount,
        SUM(fraud_count)            AS total_fraud,
        ROUND(AVG(fraud_rate_pct), 2) AS avg_fraud_rate
    FROM gold_daily_summary
""").toPandas()

top_city = spark.sql("""
    SELECT city, fraud_count, fraud_rate_pct
    FROM gold_fraud_by_city
    ORDER BY fraud_rate_pct DESC
    LIMIT 1
""").toPandas()

top_txn_type = spark.sql("""
    SELECT transaction_type, fraud_count, fraud_rate_pct
    FROM gold_fraud_by_type
    ORDER BY fraud_rate_pct DESC
    LIMIT 1
""").toPandas()

top_customer = spark.sql("""
    SELECT customer_id, fraud_count, total_amount_spent
    FROM gold_high_risk_customers
    ORDER BY fraud_count DESC
    LIMIT 1
""").toPandas()

print("=" * 55)
print("       BANKGUARD — DAILY FRAUD REPORT")
print(f"       Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 55)
print(f"""
OVERALL SUMMARY
───────────────────────────────────────────────────────
Total Transactions  : {int(daily_summary['total_txns'][0]):,}
Total Amount        : ₹{daily_summary['total_amount'][0]:,.2f}
Total Fraud Cases   : {int(daily_summary['total_fraud'][0])}
Average Fraud Rate  : {daily_summary['avg_fraud_rate'][0]}%

HIGHEST RISK CITY
───────────────────────────────────────────────────────
City                : {top_city['city'][0]}
Fraud Cases         : {int(top_city['fraud_count'][0])}
Fraud Rate          : {top_city['fraud_rate_pct'][0]}%

RISKIEST TRANSACTION TYPE
───────────────────────────────────────────────────────
Type                : {top_txn_type['transaction_type'][0]}
Fraud Cases         : {int(top_txn_type['fraud_count'][0])}
Fraud Rate          : {top_txn_type['fraud_rate_pct'][0]}%

HIGHEST RISK CUSTOMER
───────────────────────────────────────────────────────
Customer ID         : {top_customer['customer_id'][0]}
Fraud Transactions  : {int(top_customer['fraud_count'][0])}
Total Spent         : ₹{top_customer['total_amount_spent'][0]:,.2f}
""")
print("=" * 55)
print("Report complete. All metrics sourced from Gold layer.")
print("=" * 55)